*Week 6 Homework*

***Search for APIs related to your final project and list them in a markdown cell (if any! if there are none, make note of this, but try to find some resources that are adjacent to your project. If there are no directly related APIs, this may be a strong case for web scraping which we will cover in the coming weeks)***

NYC Open Data Socrata API: https://www.nyc.gov/opendata/get-started/user-guides/open-data-apis
This API can provide programmatic access to NYC datasets, including potentially relevant public health and emergency department data. I will investigate whether the influenza-like illness dataset has an API endpoint and use Python to retrieve and filter records.
#
New York State Open Data: https://data.ny.gov/developers
New York State provides API access to datasets published on its open data platform. I will investigate whether the 2022 health insurance enrollment and demographic datasets relevant to my project are available through this platform.
#
NYC Department of Health Public Use Death Datasets: https://www.nyc.gov/site/doh/data/data-sets/public-use-death-datasets.page 
This resource provides mortality data by geographic area and demographic characteristics. The listed public-use datasets are Excel files rather than necessarily direct API resources, so I will investigate whether equivalent data are available through NYC Open Data.
#
U.S. Census Bureau API: https://www.census.gov/data/developers/data-sets.html 
This API can provide population and socioeconomic estimates that may help me calculate mortality and illness rates and investigate relationships with poverty and health insurance coverage. 

***Find a website (or set of websites) that contains information related to your topic that you could build a dataset from by scraping, which we will explore next week.***

The NYC Neighborhood Atlas offers insights to social determinants of health
https://www.health.ny.gov/diseases/chronic/  

***Use the MET API to assemble a new list of dictionaries containing only items that relate to a specific animal, person, place. This doesn’t have to be comprehensive, as you may end up with thousands of items, so make a list that has at least 5 items in it.***

**Cat API**

In [20]:
##setting up search for cat API
base_url = "https://collectionapi.metmuseum.org"
search_path = "/public/collection/v1.1/search"
search_query = "?q=Cat"

url = f"{base_url}{search_path}{search_query}"

print(url)

https://collectionapi.metmuseum.org/public/collection/v1.1/search?q=Cat


In [34]:
print(type(search_response))
print(dir(search_response))

<class 'requests.models.Response'>
['__annotations__', '__attrs__', '__bool__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__enter__', '__eq__', '__exit__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__nonzero__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_content', '_content_consumed', '_next', 'apparent_encoding', 'close', 'connection', 'content', 'cookies', 'elapsed', 'encoding', 'headers', 'history', 'is_permanent_redirect', 'is_redirect', 'iter_content', 'iter_lines', 'json', 'links', 'next', 'ok', 'raise_for_status', 'raw', 'reason', 'request', 'status_code', 'text', 'url']


In [14]:
search_response = requests.get(url)

print(type(search_response))
print(search_response.ok)
print(search_response.status_code)

<class 'requests.models.Response'>
True
200


In [15]:
# confirming data recieved 
if search_response.ok:
    print("Data received!")
else:
    print("Something went wrong:", search_response.status_code)

Data received!


In [33]:
public_names = []

for name in dir(search_response):
    if name[0] != "_":
        public_names.append(name)

print(public_names)

['apparent_encoding', 'close', 'connection', 'content', 'cookies', 'elapsed', 'encoding', 'headers', 'history', 'is_permanent_redirect', 'is_redirect', 'iter_content', 'iter_lines', 'json', 'links', 'next', 'ok', 'raise_for_status', 'raw', 'reason', 'request', 'status_code', 'text', 'url']


In [16]:
## Getting object IDs 
parsed_json_data = search_response.json()

print(type(parsed_json_data))
print(parsed_json_data.keys())

<class 'dict'>
dict_keys(['total', 'objectIDs'])


In [17]:

print("Total matches:", parsed_json_data["total"])

print("Number of IDs returned:",
      len(parsed_json_data["objectIDs"] or []))

Total matches: 1814
Number of IDs returned: 100


In [18]:
object_ids = parsed_json_data["objectIDs"] or []

first_found_item = object_ids[0]

print(first_found_item)

91120


In [22]:
##list of dictionaries and then setting to 5 for my list.

cat_artworks = []

for object_id in object_ids:
    object_path = "/public/collection/v1/objects/"
    url = f"{base_url}{object_path}{object_id}"

    response_object = requests.get(url)

    if response_object.ok:
        object_data = response_object.json()
        title = object_data.get("title") or ""

        if "cat" in title.lower():
            cat_dict = {
                "objectID": object_data.get("objectID"),
                "title": title,
                "date": object_data.get("objectDate"),
                "artist": object_data.get("artistDisplayName"),
                "department": object_data.get("department"),
                "medium": object_data.get("medium")
            }

            cat_artworks.append(cat_dict)

    if len(cat_artworks) >= 5:
        break

print("Number of cat artworks:", len(cat_artworks))
print(cat_artworks)

Number of cat artworks: 5
[{'objectID': 552027, 'title': 'Cat', 'date': '664–30 BCE', 'artist': '', 'department': 'Egyptian Art', 'medium': 'Cupreous metal'}, {'objectID': 570722, 'title': 'Cat', 'date': '664–30 BCE', 'artist': '', 'department': 'Egyptian Art', 'medium': 'Cupreous metal'}, {'objectID': 570749, 'title': 'Cat', 'date': '664–30 BCE', 'artist': '', 'department': 'Egyptian Art', 'medium': 'Cupreous metal'}, {'objectID': 552028, 'title': 'Cat', 'date': '664–30 BCE', 'artist': '', 'department': 'Egyptian Art', 'medium': 'Cupreous metal'}, {'objectID': 570737, 'title': 'Cat', 'date': '664–30 BCE', 'artist': '', 'department': 'Egyptian Art', 'medium': 'Cupreous metal'}]


In [24]:
## using pandas to get a table display for my list

import pandas as pd

cat_df = pd.DataFrame(cat_artworks)
cat_df

,objectID,title,date,artist,department,medium
0,552027,Cat,664–30 BCE,,Egyptian Art,Cupreous metal
1,570722,Cat,664–30 BCE,,Egyptian Art,Cupreous metal
2,570749,Cat,664–30 BCE,,Egyptian Art,Cupreous metal
3,552028,Cat,664–30 BCE,,Egyptian Art,Cupreous metal
4,570737,Cat,664–30 BCE,,Egyptian Art,Cupreous metal
